In [ ]:
import os
import pandas as pd
import numpy as np
import networkx as nx
import warnings
import time
import json
from pathlib import Path
from gurobipy import Model, GRB, quicksum
import scipy.stats as st
warnings.filterwarnings('ignore')


In [ ]:

BASE = Path(os.environ.get("KEP_DATA_DIR", "../../data"))
POOL_DIR     = BASE / 'supplementary' / 'pool_simulations_cauonly_oversampled'
MATRICES_DIR = BASE / 'supplementary' / 'pool_matrices_cauonly_oversampled'
RESULTS_DIR  = BASE / 'supplementary' / 'simulation_results'
POOL_DIR.mkdir(parents=True, exist_ok=True)
MATRICES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_SIMS = 100
LOCI = ['A', 'B', 'C', 'DR', 'DQ']
EPLET_CLASSES = ['ClassI', 'DR', 'DQ']
ETHCATS = [1, 2, 4, 5, 6, 7]
TARGET_POOL_SIZE = 1087  


df_pat_all = pd.read_csv(BASE / 'df_receptores_imputados_final.csv', low_memory=False)
df_don_all = pd.read_csv(BASE / 'df_donantes_imputados_final.csv', low_memory=False)

df_pat_all['WL_ID_CODE'] = df_pat_all['WL_ID_CODE'].astype('int64')
df_pat_all['ETHCAT'] = pd.to_numeric(df_pat_all['ETHCAT'], errors='coerce')
df_don_all['ETHCAT_DON'] = pd.to_numeric(df_don_all['ETHCAT_DON'], errors='coerce')


df_pat = df_pat_all[df_pat_all['ETHCAT'] == 1].reset_index(drop=True).copy()
df_don = df_don_all[df_don_all['ETHCAT_DON'] == 1].reset_index(drop=True).copy()

print(f'Caucasian patients: {len(df_pat)} (out of {len(df_pat_all)} total)')
print(f'Caucasian donors:   {len(df_don)} (out of {len(df_don_all)} total)')
print(f'Target pool size:   {TARGET_POOL_SIZE}')

SIM_PARAMS = {
    'TOTAL_TIME':       10 * 12,
    'ARRIVAL_RATE':     1000 / (10 * 12),
    'MEAN_PATIENCE':    65.1552,   
    'MATCH_RUN':        3,
    'WARMUP_MONTHS':    60,   # warm-up
    'MAX_CYCLE_LENGTH': 3,
    'SEED_BASE':        42,
    'P':                1100,
    'k_opt': {'antigen': 0, 'allele': 0, 'eplet': 0},
    'Z':     {'antigen': 10, 'allele': 10, 'eplet': 140},
}
print(f"SIM_PARAMS: P={SIM_PARAMS['P']}, k_opt={SIM_PARAMS['k_opt']}, Z={SIM_PARAMS['Z']}")

RESOLUTIONS = ('antigen', 'allele', 'eplet')


In [ ]:

import re


def parse_antibody(code):
    s = str(int(code))
    n = len(s)
    if n <= 2: return ('antigen', s.zfill(2))
    elif n == 3: return ('allele', f'0{s[0]}:{s[1:]}')
    elif n == 4: return ('allele', f'{s[:2]}:{s[2:]}')
    elif n == 5: return ('allele', f'0{s[0]}:{s[1:3]}')
    else: return ('allele', f'{s[:2]}:{s[2:4]}')

df_unacc = pd.read_parquet(BASE / 'unacc_filtered.parquet')
antibodies = {}
for wid, group in df_unacc.groupby('WL_ID_CODE'):
    s = set()
    for ant, loc in zip(group['ANTIGEN'].astype(int).values, group['LOCUS'].values):
        level, value = parse_antibody(ant)
        s.add((loc, level, value))
    antibodies[int(wid)] = s
print(f'Antibodies loaded for {len(antibodies)} patients')


_ABO_COMPAT = {
    ('O','O'), ('O','A'), ('O','B'), ('O','AB'),
    ('A','A'), ('A','AB'), ('B','B'), ('B','AB'), ('AB','AB'),
}
def abo_compatible(d_abo, p_abo):
    return (d_abo, p_abo) in _ABO_COMPAT


PAT_COLS = {'A': ('A1_pat','A2_pat'), 'B': ('B1_pat','B2_pat'), 'C': ('C1_pat','C2_pat'),
            'DR': ('DR1_pat','DR2_pat'), 'DQ': ('DQ1_pat','DQ2_pat')}
DON_COLS = {'A': ('DA1','DA2'), 'B': ('DB1','DB2'), 'C': ('DC1','DC2'),
            'DR': ('DDR1','DDR2'), 'DQ': ('DDQ1','DDQ2')}




N_IMPUTATIONS = 45 

def parse_imputation_string(imp_str):
 
    result = {f'{L}{n}': None for L in ('A','B','C','DR','DQ') for n in (1,2)}
    if not isinstance(imp_str, str): return result
    tokens = imp_str.replace('+', '').split()
    locus_alleles = {'A': [], 'B': [], 'C': [], 'DR': [], 'DQ': []}
    for tok in tokens:
        if '*' not in tok: continue
        locus_raw, allele = tok.split('*', 1)
        if locus_raw == 'A': locus_alleles['A'].append(allele)
        elif locus_raw == 'B': locus_alleles['B'].append(allele)
        elif locus_raw == 'C': locus_alleles['C'].append(allele)
        elif locus_raw in ('DRB1', 'DR'): locus_alleles['DR'].append(allele)
        elif locus_raw in ('DQB1', 'DQ'): locus_alleles['DQ'].append(allele)
    for L in ('A', 'B', 'C', 'DR', 'DQ'):
        if len(locus_alleles[L]) >= 1: result[f'{L}1'] = locus_alleles[L][0]
        if len(locus_alleles[L]) >= 2: result[f'{L}2'] = locus_alleles[L][1]
        elif len(locus_alleles[L]) == 1: result[f'{L}2'] = locus_alleles[L][0] 

def sample_hla_imputation(row, rng):
    
    imps, liks = [], []
    for i in range(1, N_IMPUTATIONS + 1):
        imp = row.get(f'Imputacion_{i}')
        lik = row.get(f'Likelihood_{i}')
        if pd.notna(imp) and pd.notna(lik):
            imps.append(imp)
            liks.append(float(lik))
    if not imps:
        return None
    weights = np.array(liks)
    weights = weights / weights.sum()
    idx = rng.choice(len(imps), p=weights)
    return parse_imputation_string(imps[idx])

def fixed_hla_patient(row):
    
    return {f'{L}{n}': row.get(f'{L}{n}') for L in ('A','B','C','DR','DQ') for n in (1,2)}

def assign_hla_to_pair_row(pair_dict, hla_dict, side='pat'):
    
    col_map = PAT_COLS if side == 'pat' else DON_COLS
    for L, (c1, c2) in col_map.items():
        pair_dict[c1] = hla_dict.get(f'{L}1')
        pair_dict[c2] = hla_dict.get(f'{L}2')


def donor_dsa_triggers_from_hla(hla_dict):
    
    out = set()
    for L in ('A','B','C','DR','DQ'):
        for n in (1, 2):
            v = hla_dict.get(f'{L}{n}')
            if pd.isna(v) or v in (None, '', 'nan'): continue
            full = str(v)
            ff = full.split(':')[0].zfill(2)
            out.add((L, 'antigen', ff))
            out.add((L, 'allele', full))
    return out




def donor_dsa_triggers(row_don):
    
    for L, (c1, c2) in DON_COLS.items():
        for c in (c1, c2):
            v = row_don.get(c)
            if pd.isna(v) or v in (None, '', 'nan'): continue
            full = str(v)
            ff = full.split(':')[0].zfill(2)
            out.add((L, 'antigen', ff))
            out.add((L, 'allele', full))
    return out

def hla_pat_str(row):
    
    PMAP = {'A':'A','B':'B','C':'C','DR':'DRB1','DQ':'DQB1'}
    parts = []
    for L, (c1, c2) in PAT_COLS.items():
        for c in (c1, c2):
            v = row.get(c)
            if pd.notna(v) and v not in ('','nan'):
                parts.append(f'{PMAP[L]}*{v}')
    return ' '.join(parts)

def hla_don_str(row):
    
    PMAP = {'A':'A','B':'B','C':'C','DR':'DRB1','DQ':'DQB1'}
    parts = []
    for L, (c1, c2) in DON_COLS.items():
        for c in (c1, c2):
            v = row.get(c)
            if pd.notna(v) and v not in ('','nan'):
                parts.append(f'{PMAP[L]}*{v}')
    return ' '.join(parts)


In [ ]:
# POOL FORMATION
def hla_pat_str_from_dict(d):
    PMAP = {'A':'A','B':'B','C':'C','DR':'DRB1','DQ':'DQB1'}
    parts = []
    for L in ('A','B','C','DR','DQ'):
        for n in (1, 2):
            v = d.get(f'{L}{n}')
            if pd.notna(v) and v not in ('','nan'):
                parts.append(f'{PMAP[L]}*{v}')
    return ' '.join(parts)

def hla_don_str_from_dict(d):
    return hla_pat_str_from_dict(d)

def is_incompatible_pair(p_abo, d_abo, p_antibodies, d_dsa_triggers):
   
    if pd.isna(p_abo) or pd.isna(d_abo):
        return False
    if not abo_compatible(d_abo, p_abo):
        return True
    if p_antibodies and (d_dsa_triggers & p_antibodies):
        return True
    return False

def form_pool_cauonly(sim_id, target_size=TARGET_POOL_SIZE):
   
    rng = np.random.default_rng(100 + sim_id)

    
    n_pat = len(df_pat)
    n_don = len(df_don)

    
    n_pat_slots = max(target_size * 2, n_pat)
    n_don_slots = max(target_size * 2, n_don)

    
    pat_slot_idx = list(range(n_pat)) + list(rng.integers(0, n_pat, size=n_pat_slots - n_pat))
    don_slot_idx = list(range(n_don)) + list(rng.integers(0, n_don, size=n_don_slots - n_don))
    rng.shuffle(pat_slot_idx)
    rng.shuffle(don_slot_idx)

  
    pat_slots = []  
    for slot_id, pat_idx in enumerate(pat_slot_idx):
        row = df_pat.iloc[pat_idx]
        hla = sample_hla_imputation(row, rng)
        if hla is None:
            hla = fixed_hla_patient(row)
        pat_slots.append({
            'slot_id': slot_id,
            'WL_ID_CODE': int(row['WL_ID_CODE']),
            'ABO_pat': row['ABO'],
            'antibodies': antibodies.get(int(row['WL_ID_CODE']), set()),
            'hla': hla,
        })

    don_slots = []
    for slot_id, don_idx in enumerate(don_slot_idx):
        row = df_don.iloc[don_idx]
        hla = sample_hla_imputation(row, rng)
        if hla is None:
            hla = {f'{L}{n}': row.get(f'D{L}{n}' if L != 'A' else f'DA{n}') for L in ('A','B','C','DR','DQ') for n in (1,2)}
        d_triggers = donor_dsa_triggers_from_hla(hla)
        don_slots.append({
            'slot_id': slot_id,
            'DONOR_ID': int(row.get('DONOR_ID', 0) or 0),
            'ABO_don': row['ABO_DON'],
            'hla': hla,
            'd_triggers': d_triggers,
        })

   
    pairs_rows = []
    used_pats = set()
    used_dons = set()
    for ps in pat_slots:
        if len(pairs_rows) >= target_size: break
        if ps['slot_id'] in used_pats: continue
        for ds in don_slots:
            if ds['slot_id'] in used_dons: continue
            if is_incompatible_pair(ps['ABO_pat'], ds['ABO_don'], ps['antibodies'], ds['d_triggers']):
                # Build pair row
                pair = {
                    'WL_ID_CODE': ps['WL_ID_CODE'],
                    'DONOR_ID': ds['DONOR_ID'],
                    'pat_slot_id': ps['slot_id'],
                    'don_slot_id': ds['slot_id'],
                    'ABO_pat': ps['ABO_pat'],
                    'ABO_don': ds['ABO_don'],
                    'ETHCAT': 1,
                }
                assign_hla_to_pair_row(pair, ps['hla'], side='pat')
                assign_hla_to_pair_row(pair, ds['hla'], side='don')
                pairs_rows.append(pair)
                used_pats.add(ps['slot_id'])
                used_dons.add(ds['slot_id'])
                break

    return pd.DataFrame(pairs_rows)


In [ ]:
# GENERATE 100 SIM POOLS 

OVERWRITE_POOLS = False 

t0 = time.time()
for sim_id in range(N_SIMS):
    out_path = POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet'
    if out_path.exists() and not OVERWRITE_POOLS:
        continue
    pool_df = form_pool_cauonly(sim_id)
    pool_df.to_parquet(out_path, compression='snappy')
    if (sim_id + 1) % 10 == 0 or sim_id == 0:
        print(f'  sim {sim_id+1:3d}/{N_SIMS}: pool size = {len(pool_df)}  ({(time.time()-t0)/60:.1f} min elapsed)')

print(f'\nAll pools generated. Total time: {(time.time()-t0)/60:.1f} min')


sizes = []
for sim_id in range(N_SIMS):
    df = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    sizes.append(len(df))
print(f'Pool size — mean: {np.mean(sizes):.0f}, std: {np.std(sizes):.1f}, min: {min(sizes)}, max: {max(sizes)}')


In [ ]:
# PRECOMPUTE COMPAT + MISMATCH MATRICES per sim


OVERWRITE_MATRICES = False


import os, sys
HANS_DIR = BASE / 'codigo_Hans_2' / 'ET_eplet_calculator-master'
src_yaml = HANS_DIR / 'simulator/sim_yamls/sim_settings_test_eplet.yaml'
patched_yaml = HANS_DIR / 'simulator/sim_yamls/_patched_settings.yaml'
with open(src_yaml) as f: text = f.read()
old_prefix = '/Users/vale/Library/CloudStorage/OneDrive-UniversidadAdolfoIbanez/TESIS/CODIGO/MISMATCH EPLET EPREGISTRY/codigo_Hans_2/ET_eplet_calculator-master/'
new_prefix = str(HANS_DIR) + '/'
patched_yaml.write_text(text.replace(old_prefix, new_prefix))

orig_cwd = os.getcwd()
os.chdir(HANS_DIR)
if str(HANS_DIR) not in sys.path: sys.path.insert(0, str(HANS_DIR))

import simulator.magic_values.etkidney_simulator_settings as es
from simulator.code.HLA.HLASystem import HLASystem, HLAProfile
from simulator.code.HLA.EpletSystem import EpletSystem
import simulator.code.utils.read_input_files as rdr
import simulator.magic_values.magic_values_rules as mgr

ss = rdr.read_sim_settings(str(patched_yaml))
ss.NEEDED_SPLIT_MISMATCHES = [mgr.HLA_A, mgr.HLA_B, mgr.HLA_C, mgr.HLA_DR, mgr.HLA_DQB]
ss.NEEDED_BROAD_MISMATCHES = [mgr.HLA_A, mgr.HLA_B, mgr.HLA_C, mgr.HLA_DR, mgr.HLA_DQB]
print('Initializing HLA + Eplet systems...')
hla_system = HLASystem(ss)
eplet_system = EpletSystem(ss, hla_system)
os.chdir(orig_cwd)
print('  done')

LOCUS_PREFIX = {'A': 'A', 'B': 'B', 'C': 'C', 'DR': 'DRB1', 'DQ': 'DQB1'}

def allele_mm_locus(d_alleles, p_alleles):
    remaining = list(p_alleles); mm = 0
    for d in d_alleles:
        if d in remaining: remaining.remove(d)
        else: mm += 1
    return mm

def build_compat_matrix(sim_df):
    n = len(sim_df)
    pat_abo = sim_df['ABO_pat'].values
    don_abo = sim_df['ABO_don'].values
    pat_ids = sim_df['WL_ID_CODE'].values
    pat_antibodies = [antibodies.get(int(pat_ids[i]), set()) for i in range(n)]
    don_triggers = [donor_dsa_triggers(sim_df.iloc[i]) for i in range(n)]
    compat = np.zeros((n, n), dtype=np.int8)
    for i in range(n):
        for j in range(n):
            if i == j: continue
            if abo_compatible(don_abo[j], pat_abo[i]) and not (don_triggers[j] & pat_antibodies[i]):
                compat[i, j] = 1
    return compat

def build_allele_mm_per_locus(sim_df, compat):
    n = len(sim_df)
    out = {L: np.zeros((n, n), dtype=np.int8) for L in LOCI}
    for L in LOCI:
        c1p, c2p = PAT_COLS[L]; c1d, c2d = DON_COLS[L]
        pa = [[sim_df.iloc[i][c1p], sim_df.iloc[i][c2p]] for i in range(n)]
        da = [[sim_df.iloc[i][c1d], sim_df.iloc[i][c2d]] for i in range(n)]
        for i in range(n):
            for j in range(n):
                if compat[i, j] == 1:
                    out[L][i, j] = allele_mm_locus(da[j], pa[i])
    return out

def run_hans_on_compat(sim_df, compat):
    n = len(sim_df)
    pat_strings = [hla_pat_str(sim_df.iloc[i]) for i in range(n)]
    don_strings = [hla_don_str(sim_df.iloc[i]) for i in range(n)]
    pat_profiles = [HLAProfile(rdr.fix_hla_string(pd.Series([s]))[0], hla_system=hla_system) for s in pat_strings]
    don_profiles = [HLAProfile(rdr.fix_hla_string(pd.Series([s]))[0], hla_system=hla_system) for s in don_strings]
    antigen_keys = {'A':'mms_hla_a','B':'mms_hla_b','C':'mms_hla_c','DR':'mms_hla_dr','DQ':'mms_hla_dqb'}
    EPLET_CLASS_MAP = {'ClassI':'I','DR':'drb1345','DQ':'dq'}
    antigen_mat = {L: np.zeros((n, n), dtype=np.int8) for L in LOCI}
    eplet_mat   = {cls: np.zeros((n, n), dtype=np.int16) for cls in EPLET_CLASS_MAP}
    for i in range(n):
        ph = pat_profiles[i]
        js = np.where(compat[i] == 1)[0]
        for j in js:
            dh = don_profiles[j]
            try:
                mm = hla_system.count_mismatches(p_hla=ph, d_hla=dh)
                for L, key in antigen_keys.items():
                    v = mm.get(key, 0)
                    if pd.notna(v): antigen_mat[L][i, j] = int(v)
                ep_cls = eplet_system.get_epletregistry_mm_per_locus(pat_hla=ph, don_hla=dh)
                for cls_name, hans_key in EPLET_CLASS_MAP.items():
                    v = ep_cls.get(hans_key, 0)
                    if pd.notna(v): eplet_mat[cls_name][i, j] = int(v)
            except Exception:
                pass
    return antigen_mat, eplet_mat

def precompute_one_sim(sim_id, overwrite=False):
    sim_dir = MATRICES_DIR / f'sim_{sim_id:03d}'
    if sim_dir.exists() and not overwrite:
        if (sim_dir / 'compatibility.parquet').exists():
            return False
    sim_dir.mkdir(parents=True, exist_ok=True)
    sim_df = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    compat = build_compat_matrix(sim_df)
    pd.DataFrame(compat).to_parquet(sim_dir / 'compatibility.parquet', compression='snappy')
    allele = build_allele_mm_per_locus(sim_df, compat)
    for L, mat in allele.items():
        pd.DataFrame(mat).to_parquet(sim_dir / f'mismatch_allele_{L}.parquet', compression='snappy')
    antigen, eplet = run_hans_on_compat(sim_df, compat)
    for L, mat in antigen.items():
        pd.DataFrame(mat).to_parquet(sim_dir / f'mismatch_antigen_{L}.parquet', compression='snappy')
    for cls, mat in eplet.items():
        pd.DataFrame(mat).to_parquet(sim_dir / f'mismatch_eplet_{cls}.parquet', compression='snappy')
    return True

t0 = time.time()
for sim_id in range(N_SIMS):
    did = precompute_one_sim(sim_id, overwrite=OVERWRITE_MATRICES)
    if did and ((sim_id + 1) % 5 == 0 or sim_id == 0):
        print(f'  sim {sim_id+1:3d}/{N_SIMS}: matrices done ({(time.time()-t0)/60:.1f} min elapsed)')


In [ ]:
import os
import pandas as pd
import numpy as np
from collections import Counter


pools = []
for sim_id in range(N_SIMS):
    p = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    pools.append(p)


sizes = [len(p) for p in pools]
print(f'━━━ POOL SIZE ━━━')
print(f'  min={min(sizes)}, max={max(sizes)}, mean={np.mean(sizes):.1f}, std={np.std(sizes):.2f}')


unique_pat = [p['WL_ID_CODE'].nunique() for p in pools]
unique_don = [p['DONOR_ID'].nunique() for p in pools]
print(f'\n━━━ DIVERSITY PER SIM ━━━')
print(f'  Unique patients per sim: min={min(unique_pat)}, max={max(unique_pat)}, mean={np.mean(unique_pat):.1f}')
print(f'  Unique donors per sim:   min={min(unique_don)}, max={max(unique_don)}, mean={np.mean(unique_don):.1f}')
print(f'  (there are {len(df_pat)} Cau patients, {len(df_don)} Cau donors)')


dup_pat = []  
dup_don = []
max_copies = []
for p in pools:
    cnt = p['WL_ID_CODE'].value_counts()
    dup_pat.append((cnt > 1).sum())
    dup_don.append((p['DONOR_ID'].value_counts() > 1).sum())
    max_copies.append(cnt.max())
print(f'\n━━━ OVERSAMPLING ━━━')
print(f'  Patients with >1 copy per sim: mean={np.mean(dup_pat):.1f}')
print(f'  Donors with >1 copy per sim:   mean={np.mean(dup_don):.1f}')
print(f'  Max copies of the same patient: {max(max_copies)}')


combined = pd.concat(pools, ignore_index=True)
print(f'\n━━━ POOL ABO (across all sims) ━━━')
print(f'  Donor ABO: {dict(combined["ABO_don"].value_counts())}')
print(f'  Patient ABO: {dict(combined["ABO_pat"].value_counts())}')


ABO_OK = {('O','O'),('O','A'),('O','B'),('O','AB'),('A','A'),('A','AB'),('B','B'),('B','AB'),('AB','AB')}
combined['abo_compat'] = combined.apply(lambda r: (r['ABO_don'], r['ABO_pat']) in ABO_OK, axis=1)
n_abo_inc = (~combined['abo_compat']).sum()
n_dsa = combined['abo_compat'].sum()
print(f'\n━━━ REASON FOR INCOMPATIBILITY ━━━')
print(f'  ABO incompatible:        {n_abo_inc:,} ({n_abo_inc/len(combined)*100:.1f}%)')
print(f'  ABO compat + DSA hit:    {n_dsa:,} ({n_dsa/len(combined)*100:.1f}%)')


pair_keys = list(zip(combined['WL_ID_CODE'], combined['DONOR_ID']))
unique_pair_combos = len(set(pair_keys))
print(f'\n━━━ OVERLAP BETWEEN SIMS ━━━')
print(f'  Total pair-instances:      {len(combined):,}')
print(f'  Unique (pat, don) combos:  {unique_pair_combos:,}')
print(f'  Repetition rate:           {1 - unique_pair_combos/len(combined):.1%}')


sim_count_per_pair = Counter(pair_keys)
how_many_sims_dist = Counter(sim_count_per_pair.values())


p0 = pools[0]
hla_pat = p0[['A1_pat','A2_pat','B1_pat','B2_pat','C1_pat','C2_pat','DR1_pat','DR2_pat','DQ1_pat','DQ2_pat']].astype(str).apply(tuple, axis=1)
hla_don = p0[['DA1','DA2','DB1','DB2','DC1','DC2','DDR1','DDR2','DDQ1','DDQ2']].astype(str).apply(tuple, axis=1)
print(f'\n━━━ HLA DIVERSITY (sim 0) ━━━')
print(f'  Unique patient HLA profiles: {hla_pat.nunique():,} of {len(p0):,} pairs ({hla_pat.nunique()/len(p0):.1%})')
print(f'  Unique donor HLA profiles:   {hla_don.nunique():,} of {len(p0):,} pairs ({hla_don.nunique()/len(p0):.1%})')

df_pat_orig = pd.read_csv(BASE / 'df_receptores_imputados_final.csv', low_memory=False)
if 'END_CPRA' in df_pat_orig.columns:
    cau_pat_ids = set(df_pat_orig[df_pat_orig['ETHCAT'] == 1]['WL_ID_CODE'].astype('int64'))
    hs_ids = set(df_pat_orig[(df_pat_orig['ETHCAT'] == 1) & (df_pat_orig['END_CPRA'] > 80)]['WL_ID_CODE'].astype('int64'))
    hs_per_sim_count = [p['WL_ID_CODE'].isin(hs_ids).sum() for p in pools]
    hs_per_sim_unique = [p[p['WL_ID_CODE'].isin(hs_ids)]['WL_ID_CODE'].nunique() for p in pools]


In [ ]:
# PER-SIM DATA LOADER

def load_sim_data(sim_id):
    pool_df = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    sim_dir = MATRICES_DIR / f'sim_{sim_id:03d}'
    compat = pd.read_parquet(sim_dir / 'compatibility.parquet').values.astype(np.int8)
    antigen_mm = {L: pd.read_parquet(sim_dir / f'mismatch_antigen_{L}.parquet').values for L in LOCI}
    allele_mm  = {L: pd.read_parquet(sim_dir / f'mismatch_allele_{L}.parquet').values for L in LOCI}
    eplet_mm   = {cls: pd.read_parquet(sim_dir / f'mismatch_eplet_{cls}.parquet').values for cls in EPLET_CLASSES}
    return {'pool_df': pool_df, 'compat': compat,
            'antigen_mm': antigen_mm, 'allele_mm': allele_mm, 'eplet_mm': eplet_mm}

# Quick test
sd = load_sim_data(0)


In [ ]:
# BUILD WEIGHT MATRICES 
MAX_ANTIGEN_10LOCI = 10
MAX_ALLELE_10LOCI  = 10
MAX_EPLET_10LOCI   = 140

def build_weights_10loci(sim_data):
    am = sim_data['antigen_mm']; al = sim_data['allele_mm']; ep = sim_data['eplet_mm']
    sum_antigen = sum(am[L] for L in LOCI).astype(np.int32)
    sum_allele  = sum(al[L] for L in LOCI).astype(np.int32)
    sum_eplet   = (ep['ClassI'] + ep['DR'] + ep['DQ']).astype(np.int32)
    return {
        'antigen': (MAX_ANTIGEN_10LOCI - sum_antigen).astype(np.int32),
        'allele':  (MAX_ALLELE_10LOCI  - sum_allele).astype(np.int32),
        'eplet':   (MAX_EPLET_10LOCI   - sum_eplet).astype(np.int32),
        'score_classI': (6 - (am['A'] + am['B'] + am['C'])).astype(np.int32),
        'score_DR':     (2 - am['DR']).astype(np.int32),
        'score_DQ':     (2 - am['DQ']).astype(np.int32),
    }


In [ ]:
# GRAPH, WEIGHT-ATTACH, OPTIMIZATION 

def create_graph(waiting_indices, compat):
    G = nx.DiGraph()
    G.add_nodes_from(waiting_indices)
    for i in waiting_indices:
        for j in waiting_indices:
            if i == j: continue
            if compat[i, j] == 1:
                G.add_edge(j, i)
    return G

def changing_resolution_weights(G, weight_matrix):
    for u, v in G.edges():
        G[u][v]['weight'] = int(weight_matrix[v, u])

def optimization(G, l=3, k_quality=0, Z=10, P=1100):
    total_cycles = list(nx.simple_cycles(G, length_bound=l))
    valid_cycles = [c for c in total_cycles
                    if all(G[u][v]['weight'] >= k_quality
                           for u, v in zip(c, c[1:] + c[:1]))]
    G_opt = nx.DiGraph()
    if not valid_cycles:
        return G_opt, []
    m = Model('kep'); m.setParam('OutputFlag', 0)
    x = {tuple(c): m.addVar(vtype=GRB.BINARY) for c in valid_cycles}
    m.setObjective(
        quicksum(
            x[tuple(c)] * (
                (len(c) + (1.0 / P) * sum(G[u][v]['weight'] / Z
                                          for u, v in zip(c, c[1:] + c[:1])))
                / P
            )
            for c in valid_cycles
        ),
        GRB.MAXIMIZE
    )
    for node in G.nodes():
        m.addConstr(quicksum(x[tuple(c)] for c in valid_cycles if node in c) <= 1)
    m.optimize()
    selected = []
    if m.status == GRB.OPTIMAL:
        for c in valid_cycles:
            if x[tuple(c)].X > 0.5:
                selected.append(c)
                for i in range(len(c)):
                    u, v = c[i], c[(i + 1) % len(c)]
                    G_opt.add_edge(u, v, weight=G[u][v]['weight'])
    return G_opt, selected


In [ ]:
# RUN ONE SIMULATION 

def run_simulation(sim_id, opt_resolution, sim_data, weights, params):
    pool_df = sim_data['pool_df']
    compat  = sim_data['compat']
    n = len(pool_df)

    weight_for_obj = weights[opt_resolution]
    k_opt = params['k_opt'][opt_resolution]
    Z     = params['Z'][opt_resolution]

    
    ss = np.random.SeedSequence(params['SEED_BASE'] + sim_id * 1000)
    rng_arr, rng_dep = (np.random.default_rng(s) for s in ss.spawn(2))

    available = set(range(n))
    waiting = []
    arrival_t, departure_t = {}, {}
    historial_cycles = []
    historial_departures = []
    pool_sizes = []
    deadline = {}
    runs_participated = {}
    n_arrivals = 0
    n_departures = 0

    quality = {res: [] for res in ('antigen', 'allele', 'eplet')}
    quality.update({cls: [] for cls in ('classI', 'DR', 'DQ')})

    WARMUP = params.get('WARMUP_MONTHS', 0)
    for month in range(params['TOTAL_TIME']):
        counting = month >= WARMUP
        n_arr = rng_arr.poisson(params['ARRIVAL_RATE'])
        if n_arr > 0:
            new_pairs = rng_arr.choice(list(available), size=n_arr, replace=False)
            for p in new_pairs:
                p_int = int(p)
                arrival_t[p_int] = month
                available.discard(p_int)
                waiting.append(p_int)
                deadline[p_int] = month + rng_dep.exponential(params['MEAN_PATIENCE'])
                n_arrivals += (1 if counting else 0)

        if (month + 1) % params['MATCH_RUN'] == 0 and len(waiting) >= 2:
            if counting: pool_sizes.append(len(waiting))
            for w in waiting:
                runs_participated[w] = runs_participated.get(w, 0) + 1
            G = create_graph(waiting, compat)
            changing_resolution_weights(G, weight_for_obj)
            G_opt, selected = optimization(G, l=params['MAX_CYCLE_LENGTH'],
                                              k_quality=k_opt, Z=Z, P=params['P'])
            for u, v in (G_opt.edges() if counting else []):
                quality['antigen'].append(int(weights['antigen'][v, u]))
                quality['allele'].append(int(weights['allele'][v, u]))
                quality['eplet'].append(int(weights['eplet'][v, u]))
                quality['classI'].append(int(weights['score_classI'][v, u]))
                quality['DR'].append(int(weights['score_DR'][v, u]))
                quality['DQ'].append(int(weights['score_DQ'][v, u]))
            historial_cycles.extend(selected if counting else [])
            cycled = {p for c in selected for p in c}
            waiting = [w for w in waiting if w not in cycled]
            for p_int in cycled: departure_t[int(p_int)] = month

        departed_now = [w for w in waiting if deadline[w] <= month]
        if departed_now:
            ds = set(departed_now)
            waiting = [w for w in waiting if w not in ds]
            for p_int in (departed_now if counting else []):
                historial_departures.append(p_int)
                n_departures += 1

    waiting_times = []
    for p in {p for c in historial_cycles for p in c}:
        if p in runs_participated:
            waiting_times.append(runs_participated[p])

    n_tx = sum(len(c) for c in historial_cycles)
    F_total = n_tx / max(n_arrivals, 1)
    L_total = n_departures / max(n_arrivals, 1)

    return {
        'sim_id': sim_id, 'opt_resolution': opt_resolution,
        'total_arrivals': n_arrivals, 'total_transplants': n_tx,
        'total_departures': n_departures,
        'F_total': F_total, 'L_total': L_total,
        'quality': quality, 'waiting_times': waiting_times,
        'historial_cycles': historial_cycles,
        'avg_pool_size': float(np.mean(pool_sizes)) if pool_sizes else 0.0,
    }


In [ ]:
# FULL LOOP: 100 sims x 3 resolutions

all_results = {res: [] for res in RESOLUTIONS}
t0 = time.time()

for sim_id in range(N_SIMS):
    sim_data = load_sim_data(sim_id)
    weights = build_weights_10loci(sim_data)
    for opt_res in RESOLUTIONS:
        result = run_simulation(sim_id, opt_res, sim_data, weights, SIM_PARAMS)
        all_results[opt_res].append(result)
    if (sim_id + 1) % 5 == 0 or sim_id == 0:
        
        print(f'  Sim {sim_id+1:3d}/{N_SIMS} ')

print(f'\nDone. Total time: {(time.time()-t0)/60:.1f} min')


In [ ]:
# AGGREGATION: build results table per resolution, save XLSX

def mean_ci(values, conf=0.95):
    arr = np.asarray([v for v in values if pd.notna(v)], dtype=float)
    if len(arr) < 2:
        if len(arr) == 1: return arr[0], f"{arr[0]:.3f} [-; -]"
        return float('nan'), 'nan'
    m, s = arr.mean(), arr.std(ddof=1)
    low, high = st.t.interval(conf, len(arr)-1, loc=m, scale=s/np.sqrt(len(arr)))
    return m, f"{m:.3f} [{low:.3f}; {high:.3f}]"

def build_table(results_for_res):
    rs = results_for_res
    arr_tot = np.mean([r['total_arrivals'] for r in rs])
    tx_tot  = np.mean([r['total_transplants'] for r in rs])
    F_vals = [r['F_total'] for r in rs]
    L_vals = [r['L_total'] for r in rs]
    _, F_txt = mean_ci(F_vals)
    _, L_txt = mean_ci(L_vals)
    ant_vals = [np.mean(r['quality']['antigen']) for r in rs if r['quality']['antigen']]
    all_vals = [np.mean(r['quality']['allele']) for r in rs if r['quality']['allele']]
    epl_vals = [np.mean(r['quality']['eplet']) for r in rs if r['quality']['eplet']]
    cI_vals  = [np.mean(r['quality']['classI']) for r in rs if r['quality']['classI']]
    dr_vals  = [np.mean(r['quality']['DR']) for r in rs if r['quality']['DR']]
    dq_vals  = [np.mean(r['quality']['DQ']) for r in rs if r['quality']['DQ']]
    wt_vals  = [np.mean(r['waiting_times']) for r in rs if r['waiting_times']]
    _, ant_t = mean_ci(ant_vals); _, all_t = mean_ci(all_vals); _, epl_t = mean_ci(epl_vals)
    _, cI_t  = mean_ci(cI_vals);  _, dr_t  = mean_ci(dr_vals);  _, dq_t  = mean_ci(dq_vals)
    still = round(1 - np.mean(F_vals) - np.mean(L_vals), 3)

    return pd.DataFrame([{
        'Subpopulation': 'Caucasian (entire pop)',
        'Arrivals': round(arr_tot, 2),
        'Transplants': round(tx_tot, 2),
        'F(s) (Matched)': F_txt,
        'HLA(s) Antigen': ant_t,
        'HLA(s) Allele':  all_t,
        'HLA(s) Eplets':  epl_t,
        'Waiting Time': mean_ci(wt_vals)[1],
        'Pool Size': mean_ci([r['avg_pool_size'] for r in rs])[1],
        'L(s) (Left Unmatched)': L_txt,
        '1-F(s)-L(s)': still,
        'HLA ClassI': cI_t,
        'HLA DR': dr_t,
        'HLA DQ': dq_t,
    }])

tables = {}
for opt_res in RESOLUTIONS:
    print(f"\n===== {opt_res} =====")
    tbl = build_table(all_results[opt_res])
    tables[opt_res] = tbl
    print(tbl.to_string())

out_path = RESULTS_DIR / 'results_cauonly_oversampled_10loci_3scenarios.xlsx'
with pd.ExcelWriter(out_path) as writer:
    for opt_res, tbl in tables.items():
        tbl.to_excel(writer, sheet_name=f'opt_{opt_res}', index=False)
print(f'\nSaved: {out_path}')
